# **Regression: Lasso Regression (L1 Regularization)**

## **Justification of Preprocessing Strategy**

### **The Absolute Necessity of Scaling for L1 Penalty**
**Lasso Regression** introduces an L1 regularization penalty to the Ordinary Least Squares (OLS) loss function, adding the sum of the absolute values of the coefficients to the optimization target.

Because the penalty directly restricts the absolute magnitude of the $\beta$ weights, features with larger underlying numerical ranges will naturally have smaller coefficients to compensate, making them the target of unfair penalization. To prevent the algorithm from erroneously suppressing clinically significant variables, the feature space must be uniform. We will evaluate both **Standardization** and **Normalization** across all optimization levels to discover which framework yields the most accurate predictions for the continuous `diabetes_risk_score`.

### **Automated Feature Selection and Sparsity**
The unique mathematical trait of L1 regularization is its ability to force less important feature coefficients to become **exactly zero**. In a dataset containing 100,000 samples and numerous dummy-encoded variables, Lasso acts as an embedded feature selection tool, filtering out noise and tackling multicollinearity. To avoid data leakage, we drop categorical targets (`diagnosed_diabetes`, `diabetes_stage`), and we **omit stratification** during the split since we are modeling a continuous numerical distribution.


## **Experiment Design**

We have designed a complete tournament consisting of **6 distinct runs** to evaluate both scalers across 3 optimization levels, using **MAE, RMSE, and $R^2$** as performance indicators:

* **Run 1 & 2: Lasso Baseline** — Testing the Lasso model with strict Scikit-Learn default parameters under **Standardization** vs. **Normalization**.
* **Run 3 & 4: GridSearchCV Tuning** — Performing an exhaustive search over a fixed grid of the `alpha` parameter under **Standardization** vs. **Normalization**.
* **Run 5 & 6: Optuna Optimization** — Utilizing Bayesian optimization to fine-tune both `alpha` and `max_iter` continuously under **Standardization** vs. **Normalization**.

In all optimization runs (GridSearchCV and Optuna), trials are evaluated using **3-Fold Cross-Validation** to guarantee model generalizability.


In [ ]:
import pandas as pd
import numpy as np
import time
import mlflow
import optuna
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.linear_model import Lasso
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# MLflow Configuration
mlflow.set_tracking_uri("sqlite:///C:/Users/Tiago Silva/Uni/OneDrive - Universidade Portucalense/Ambiente de Trabalho/Uni/3ano2sem/LAD/Grupo5_ProjetoLAD_Parte2/TrabalhoLAD/models/mlflow.db")
mlflow.set_experiment("Regression_Lasso")

# Data Loading and Preparation
df = pd.read_csv("C:\\Users\\Tiago Silva\\Uni\\OneDrive - Universidade Portucalense\\Ambiente de Trabalho\\Uni\\3ano2sem\\LAD\\Grupo5_ProjetoLAD_Parte2\\TrabalhoLAD\\data\\diabetes_dataset_new_variables.csv")
categorical_cols = ['gender', 'ethnicity', 'smoking_status', 'education_level', 'employment_status', 'age_groups', 'weight_status', 'income_level']
df_final = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

# Drop classification targets to avoid data leakage
X = df_final.drop(["diabetes_risk_score", "diagnosed_diabetes", "diabetes_stage"], axis=1, errors='ignore')
y = df_final['diabetes_risk_score']

# Split data (80/20) - Continuous target means NO stratification
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
num_cols = X_train.select_dtypes(include=['float64', 'int64']).columns

def log_regression_metrics(model, X_tr, y_tr, X_te, y_te, duration):
    """Logs both Train and Test metrics to evaluate Overfitting/Underfitting"""
    y_tr_pred = model.predict(X_tr)
    y_te_pred = model.predict(X_te)
    
    # Train Partition Metrics
    mlflow.log_metric("mae_train", mean_absolute_error(y_tr, y_tr_pred))
    mlflow.log_metric("rmse_train", np.sqrt(mean_squared_error(y_tr, y_tr_pred)))
    mlflow.log_metric("r2_train", r2_score(y_tr, y_tr_pred))
    
    # Test Partition Metrics
    mlflow.log_metric("mae_test", mean_absolute_error(y_te, y_te_pred))
    mlflow.log_metric("rmse_test", np.sqrt(mean_squared_error(y_te, y_te_pred)))
    mlflow.log_metric("r2_test", r2_score(y_te, y_te_pred))
    
    mlflow.log_metric("fit_time", duration)

# Scaling strategies to compare across the entire tournament
scalers = {
    "Standardization": StandardScaler(),
    "Normalization": MinMaxScaler()
}

# ---------------------------------------------------------
# RUN 1 & 2: LASSO BASELINE 
# ---------------------------------------------------------
for s_name, scaler_obj in scalers.items():
    with mlflow.start_run(run_name=f"Lasso_Baseline_{s_name}"):
        X_train_scaled = X_train.copy()
        X_test_scaled = X_test.copy()
        X_train_scaled[num_cols] = scaler_obj.fit_transform(X_train[num_cols])
        X_test_scaled[num_cols] = scaler_obj.transform(X_test[num_cols])
        
        model = Lasso(random_state=42)
        start_time = time.time()
        model.fit(X_train_scaled, y_train)
        duration = time.time() - start_time
        
        mlflow.log_param("optimization", "none_default")
        mlflow.log_param("scaler", s_name)
        mlflow.log_params(model.get_params())
        
        log_regression_metrics(model, X_train_scaled, y_train, X_test_scaled, y_test, duration)

# ---------------------------------------------------------
# RUN 3 & 4: GRIDSEARCHCV 
# ---------------------------------------------------------
param_grid = {'alpha': [0.0001, 0.001, 0.01, 0.1, 1.0, 10.0]}

for s_name, scaler_obj in scalers.items():
    with mlflow.start_run(run_name=f"Lasso_GridSearch_{s_name}"):
        X_train_scaled = X_train.copy()
        X_test_scaled = X_test.copy()
        X_train_scaled[num_cols] = scaler_obj.fit_transform(X_train[num_cols])
        X_test_scaled[num_cols] = scaler_obj.transform(X_test[num_cols])
        
        grid = GridSearchCV(
            Lasso(random_state=42, max_iter=2000), 
            param_grid, cv=3, scoring='neg_mean_absolute_error', n_jobs=-1
        )
        start_time = time.time()
        grid.fit(X_train_scaled, y_train)
        duration = time.time() - start_time
        
        best_model = grid.best_estimator_
        zero_coefs = np.sum(best_model.coef_ == 0)
        
        mlflow.log_param("optimization", "GridSearchCV")
        mlflow.log_param("scaler", s_name)
        mlflow.log_param("eliminated_features", f"{zero_coefs}/{len(best_model.coef_)}")
        mlflow.log_params(grid.best_params_)
        
        log_regression_metrics(best_model, X_train_scaled, y_train, X_test_scaled, y_test, duration)

# ---------------------------------------------------------
# RUN 5 & 6: OPTUNA 
# ---------------------------------------------------------
for s_name, scaler_obj in scalers.items():
    X_train_scaled = X_train.copy()
    X_test_scaled = X_test.copy()
    X_train_scaled[num_cols] = scaler_obj.fit_transform(X_train[num_cols])
    X_test_scaled[num_cols] = scaler_obj.transform(X_test[num_cols])
    
    def objective(trial):
        alpha_val = trial.suggest_float("alpha", 1e-5, 5.0, log=True)
        max_iter_val = trial.suggest_int("max_iter", 1000, 4000)
        model = Lasso(alpha=alpha_val, max_iter=max_iter_val, random_state=42)
        scores = -cross_val_score(model, X_train_scaled, y_train, cv=3, scoring='neg_mean_absolute_error', n_jobs=-1).mean()
        return scores.mean()

    with mlflow.start_run(run_name=f"Lasso_Optuna_{s_name}"):
        study = optuna.create_study(direction="minimize")
        start_time = time.time()
        study.optimize(objective, n_trials=15)
        duration = time.time() - start_time
        
        best_lasso_opt = Lasso(**study.best_params, random_state=42)
        best_lasso_opt.fit(X_train_scaled, y_train)
        zero_coefs_opt = np.sum(best_lasso_opt.coef_ == 0)
        
        mlflow.log_param("optimization", "optuna")
        mlflow.log_param("scaler", s_name)
        mlflow.log_param("eliminated_features", f"{zero_coefs_opt}/{len(best_lasso_opt.coef_)}")
        mlflow.log_params(study.best_params)
        
        log_regression_metrics(best_lasso_opt, X_train_scaled, y_train, X_test_scaled, y_test, duration)

[I 2026-05-20 13:29:33,078] A new study created in memory with name: no-name-25eafce3-53ca-44f4-9651-bde6de63d101
[I 2026-05-20 13:29:34,167] Trial 0 finished with value: 0.39708817103800637 and parameters: {'alpha': 0.00031441181972483923, 'max_iter': 3434}. Best is trial 0 with value: 0.39708817103800637.
[I 2026-05-20 13:29:34,987] Trial 1 finished with value: 0.4049854803690766 and parameters: {'alpha': 0.010948581690550014, 'max_iter': 2849}. Best is trial 0 with value: 0.39708817103800637.
[I 2026-05-20 13:29:35,810] Trial 2 finished with value: 0.4009191363716474 and parameters: {'alpha': 0.006350009784234054, 'max_iter': 2725}. Best is trial 0 with value: 0.39708817103800637.
[I 2026-05-20 13:29:36,656] Trial 3 finished with value: 0.39718569421438854 and parameters: {'alpha': 0.0005871830271953236, 'max_iter': 2146}. Best is trial 0 with value: 0.39708817103800637.
[I 2026-05-20 13:29:37,919] Trial 4 finished with value: 0.3970687028051461 and parameters: {'alpha': 0.000244850

## **Winner Run Selection (Priority Elimination Framework)**

### **Policy**
A run is only eligible to win if it does NOT show evidence of overfitting or underfitting. Before applying the MAE/RMSE/R² decision rules, we require the Train→Test gaps to remain small enough to indicate acceptable generalization. Runs that memorize the training set or show a large Train/Test gap are disqualified regardless of metric rank.

### **Selection Criteria (priority order)**
1. **Generalization filter (mandatory):** runs with overfitting or underfitting are removed from consideration.
2. **Priority 1 (60%): Lowest MAE (Test)** — primary objective for regression accuracy.
3. **Priority 2 (30%): Lowest RMSE (Test)** — used to reject runs where RMSE grows disproportionately relative to MAE.
4. **Priority 3 (10%): Acceptable R² (Test)** — used as a quality check to confirm the model explains the target variance well enough.
5. **Tiebreaker: Lowest Fit Time** — if MAE, RMSE, and R² are effectively tied.

### **Runs Summary**

| Run | Eliminated Features | MAE (Train) | MAE (Test) | RMSE (Train) | RMSE (Test) | R² (Train) | R² (Test) | Fit Time (s) | Gen. Filter | MAE Gap | RMSE Gap | R² Gap |
|---|---:|---:|---:|---:|---:|---:|---:|---:|:--:|---:|---:|---:|
| Lasso_Optuna_Standardization | 4/53 | 0.3966757837 | 0.4038933897 | 0.6936680357 | 0.7112917067 | 0.9941322827 | 0.9938694744 | 46.4559938908 | PASS | +0.0072176060 | +0.0176236710 | -0.0002628083 |
| Lasso_GridSearch_Standardization | 6/53 | 0.3967013490 | 0.4039317592 | 0.6936728172 | 0.7112927860 | 0.9941322018 | 0.9938694558 | 26.3300929070 | PASS | +0.0072304102 | +0.0176199688 | -0.0002627460 |
| Lasso_Optuna_Normalization | 5/53 | 0.3967142664 | 0.4039546474 | 0.6936703127 | 0.7113068971 | 0.9941322442 | 0.9938692125 | 21.1862425804 | PASS | +0.0072403810 | +0.0176365844 | -0.0002630317 |
| Lasso_GridSearch_Normalization | 12/53 | 0.3971401546 | 0.4043579826 | 0.6937018316 | 0.7112789223 | 0.9941317110 | 0.9938696947 | 4.7376437187 | PASS | +0.0072178280 | +0.0175770907 | -0.0002620163 |
| Lasso_Baseline_Standardization | 0/53 | 2.0091024164 | 2.0209383078 | 2.5151186299 | 2.5227186289 | 0.9228594445 | 0.9228848245 | 0.1960139275 | FAIL | +0.0118358914 | +0.0075999990 | +0.0000253800 |
| Lasso_Baseline_Normalization | 0/53 | 5.0415100652 | 5.0730118293 | 6.2955069526 | 6.3321558577 | 0.5166883880 | 0.5141464116 | 0.1650586128 | FAIL | +0.0315017641 | +0.0366489051 | -0.0025419764 |

### **Generalization Check (Test − Train)**
- **Lasso_Optuna_Standardization:** MAE gap = +0.0072176060, RMSE gap = +0.0176236710, R² gap = -0.0002628083 → PASS.
- **Lasso_GridSearch_Standardization:** MAE gap = +0.0072304102, RMSE gap = +0.0176199688, R² gap = -0.0002627460 → PASS.
- **Lasso_Optuna_Normalization:** MAE gap = +0.0072403810, RMSE gap = +0.0176365844, R² gap = -0.0002630317 → PASS.
- **Lasso_GridSearch_Normalization:** MAE gap = +0.0072178280, RMSE gap = +0.0175770907, R² gap = -0.0002620163 → PASS.
- **Lasso_Baseline_Standardization:** MAE gap = +0.0118358914, RMSE gap = +0.0075999990, R² gap = +0.0000253800 → FAIL.
- **Lasso_Baseline_Normalization:** MAE gap = +0.0315017641, RMSE gap = +0.0366489051, R² gap = -0.0025419764 → FAIL.

### **Step-by-Step Elimination**
**Step 1 — Apply the generalization filter**
- Passing runs: Lasso_Optuna_Standardization, Lasso_GridSearch_Standardization, Lasso_Optuna_Normalization, Lasso_GridSearch_Normalization.
- Disqualified runs: Lasso_Baseline_Standardization, Lasso_Baseline_Normalization.

**Step 2 — Compare Test MAE (Priority 1 — 60%)**
- Lasso_Optuna_Standardization: 0.4038933897
- Lasso_GridSearch_Standardization: 0.4039317592
- Lasso_Optuna_Normalization: 0.4039546474
- Lasso_GridSearch_Normalization: 0.4043579826
- Lowest MAE: **Lasso_Optuna_Standardization**.

**Step 3 — Compare Test RMSE (Priority 2 — 30%)**
- Lasso_Optuna_Standardization: 0.7112917067
- Lasso_GridSearch_Standardization: 0.7112927860
- Lasso_Optuna_Normalization: 0.7113068971
- Lasso_GridSearch_Normalization: 0.7112789223
- Lasso_Optuna_Standardization remains the best overall choice because it already leads on the primary metric and keeps an excellent RMSE.

**Step 4 — Check Test R² (Priority 3 — 10%)**
- Lasso_Optuna_Standardization: 0.9938694744
- All tuned runs are excellent, but the winner remains the strongest on the primary criterion.

### **Final Decision**
**Winner: Lasso_Optuna_Standardization**

**Justification:** Among the runs that pass the generalization filter, `Lasso_Optuna_Standardization` has the lowest Test MAE, with excellent RMSE and R² values. Fit time is not needed as a tiebreaker because there is no tie on the primary performance criteria.

## **Winner Hyperparameters**
| Parameter | Value |
|---|---|
| **alpha** | 4.427123816536028e-05 |
| **max_iter** | 1131 |
| **random_state** | 42 |
| **scaler** | Standardization |
| **eliminated_features** | 4/53 |

## **Overfitting / Underfitting Diagnosis**
- `Lasso_Baseline_Standardization` and `Lasso_Baseline_Normalization` are disqualified because their Train→Test gaps are much larger and their errors are far worse, which indicates underfitting from excessive regularization.
- The four tuned runs generalize well and are valid candidates.
- `Lasso_Optuna_Standardization` is the strongest operational choice because it leads on Test MAE while keeping the Train/Test gaps small and the fit quality very high.
- Conclusion: the selected Lasso model shows excellent generalization behavior with no signs of overfitting or underfitting.